In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.multioutput import MultiOutputRegressor, MultiOutputClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, mean_absolute_error, mean_squared_error, r2_score

In [2]:
dir = Path.cwd().parent / "data"
df_train = pd.read_parquet(dir/"train.parquet")
df_val = pd.read_parquet(dir/"test.parquet")

In [3]:
cols_drop = [
    # Textos y Metadatos
    'BEAM', 'COLUMN', 'DrawingCode', 'AceroEstructural', 'EstandarTornillos',
    'Source', 'BEAM_type', 'COLUMN_type',
    # Deterministas
    'Hole', 'Lp',
    # Calculadas
    'Capacity_Moment_Percentage', 'Capacity_Shear_Percentage', 'Capacity_Axial_Percentage'
]

p_columnas = [f'p{i}' for i in range(1, 13)]

Y1 = ["num_filas", "Bolt"]
Y2 = ['g', 'e', 'd'] + p_columnas # d y p_columnas discretos
Y3 = ["Bp"]
Y4 = ['tp', 'tws', 'ts', 'tbs', 'tst']

X1 = [c for c in df_train.columns if c not in cols_drop + Y1 + Y2 + Y3 + Y4]
X2 = [c for c in df_train.columns if c not in cols_drop + Y2 + Y3 + Y4]
X3 = [c for c in df_train.columns if c not in cols_drop + Y3 + Y4]
X4 = [c for c in df_train.columns if c not in cols_drop + Y4]

In [4]:
df_train['standard'] = df_train['standard'].astype('category')
df_val['standard'] = df_val['standard'].astype('category')

In [5]:
df_train['num_filas'] = (12 - np.sum([df_train[f'p{i}']==0 for i in range(1, 13)], axis=0))
df_val['num_filas'] = (12 - np.sum([df_val[f'p{i}']==0 for i in range(1, 13)], axis=0))

In [6]:
CATALOGO_g = np.arange(80,171, 5)
CATALOGO_e = np.arange(35, 76, 5)
CATALOGO_Bp = np.arange(140, 400, 1)
CATALOGO_tws = [0, 5, 6, 8, 10, 12, 15, 20]
CATALOGO_tp = [12, 15, 20, 25, 30, 35, 40, 45, 50, 60]
CATALOGO_ts =  [0, 10, 12, 15, 20, 25, 30, 35, 40, 60]
CATALOGO_tbs = [0, 10, 12, 15, 20, 25, 30, 35, 40, 60]
CATALOGO_tst = [0, 10, 12, 15, 20, 25, 30, 35, 40, 60]
MAPA_CATALOGOS = {
    'g': CATALOGO_g,
    'e': CATALOGO_e,
    'Bp': CATALOGO_Bp,
    'tws': CATALOGO_tws,
    'tp': CATALOGO_tp,
    'ts': CATALOGO_ts,
    'tbs': CATALOGO_tbs,
    'tst': CATALOGO_tst,
}

medias_p = {}
for col in p_columnas:
    medias_p[col] = float(df_train[df_train[col] > 0][col].median())

# -----------------------------------------------------------------------------
# Ajustar DataFrame a Catálogos Comerciales
# -----------------------------------------------------------------------------
def ajustar_df_a_catalogos(df, mapa_catalogos):
    """
    Recibe un DataFrame y un diccionario {columna: catalogo_array}.
    Devuelve un nuevo DataFrame con los valores continuos ajustados al valor
    más cercano dentro de su respectivo catálogo.
    """
    df_ajustado = df.copy()

    for col, catalogo in mapa_catalogos.items():
        if col in df_ajustado.columns:
            cat_arr = np.array(sorted(catalogo))
            vals = df_ajustado[col].values

            # Vectorización: encuentra el índice con menor distancia absoluta
            indices_cercanos = np.abs(vals[:, None] - cat_arr[None, :]).argmin(
                axis=1
            )
            df_ajustado[col] = cat_arr[indices_cercanos]

    return df_ajustado


# -----------------------------------------------------------------------------
# Enmascarar Pasos Inactivos según 'num_filas'
# -----------------------------------------------------------------------------
def aplicar_mascara_pasos(df, col_num_filas="num_filas", evaluacion = True):
    """
    Enmascara a 0.0 los pasos p_i que están por encima del número de filas activo.
    Para N filas existen N-1 pasos reales (p1 .. p_{N-1}).
    Si i >= num_filas, se fuerza p_i = 0.0.
    """
    df_mascarado = df.copy()

    if col_num_filas not in df_mascarado.columns:
        return df_mascarado

    num_filas = np.round(df_mascarado[col_num_filas].values).astype(int)
    p_cols = [
        c
        for c in df_mascarado.columns
        if c.startswith("p") and c[1:].isdigit()
    ]

    for col in p_cols:
        i = int(col[1:])  # Extrae el número '1' de 'p1', '2' de 'p2', etc.
        mask_inactivo = i >= num_filas
        df_mascarado.loc[mask_inactivo, col] = 0.0 if evaluacion else medias_p[col]

    return df_mascarado

# -----------------------------------------------------------------------------
# Evaluación de Predicciones y Catálogos
# -----------------------------------------------------------------------------
def evaluar_predicciones(df_real, df_pred):
    """
    Compara dos DataFrames columna por columna evaluando MAE, RMSE, R2 Score 
    y porcentajes de tolerancia constructiva.
    Filtra pasos inactivos p_i para evitar la inflación artificial de aciertos en ceros.
    """
    cols_comunes = [c for c in df_real.columns if c in df_pred.columns]
    df_r = df_real[cols_comunes].reset_index(drop=True)
    df_p = df_pred[cols_comunes].reset_index(drop=True)
    
    p_cols = [c for c in cols_comunes if c.startswith('p') and c[1:].isdigit()]
    resultados = []
    
    for col in cols_comunes:
        y_true = df_r[col].values
        y_pred = df_p[col].values
        
        # Filtrar pasos inactivos para evaluar sólo donde existían tornillos de verdad
        if col in p_cols:
            mask_activo = y_true > 0
            if mask_activo.sum() > 0:
                y_true = y_true[mask_activo]
                y_pred = y_pred[mask_activo]
        
        # Si después de filtrar no hay datos, saltamos la variable
        if len(y_true) == 0:
            continue
            
        error_abs = np.abs(y_true - y_pred)
        
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        r2 = r2_score(y_true, y_pred) if len(y_true) > 1 and np.std(y_true) > 0 else 1.0
        
        # Tolerancias
        acierto_exacto = (error_abs <= 0.5).mean() * 100
        pct_dentro_2mm = (error_abs <= 2.0).mean() * 100
        pct_dentro_5mm = (error_abs <= 5.0).mean() * 100
        
        resultados.append({
            'Variable': col,
            'N (Muestras)': len(y_true),
            'MAE (mm)': round(mae, 2),
            'RMSE (mm)': round(rmse, 2),
            'R2 Score': round(r2, 3),
            '% Exacto (Catálogo)': round(acierto_exacto, 1),
            '% <= 2mm (Taller)': round(pct_dentro_2mm, 1),
            '% <= 5mm (Taller)': round(pct_dentro_5mm, 1),
        })
    
    df_res = pd.DataFrame(resultados)
    print("=" * 105)
    print(" " * 20 + "EVALUACIÓN INTEGRAL DE PREDICCIONES Y TOLERANCIAS")
    print("=" * 105)
    print(df_res.to_string(index=False))
    
    return df_res

# MODELO POR CAPAS

## Entrenamiento

In [7]:
# =============================================================================
# 1. ENTRENAMIENTO EN CASCADA CON PROPAGACIÓN DE PREDICCIONES AJUSTADAS
# =============================================================================

# --- NIVEL 1 ---
m1 = MultiOutputClassifier(LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1))
m1.fit(df_train[X1], df_train[Y1])

# Predecimos sobre el TRAIN para pasárselo al Nivel 2
Y1_train_pred = pd.DataFrame(m1.predict(df_train[X1]), columns=Y1, index=df_train.index)

# --- NIVEL 2 ---
# Construimos X2 usando las PREDICCIONES del Nivel 1, no la realidad
X2_train_pred = df_train[X1].copy()
X2_train_pred[Y1] = Y1_train_pred
X2_train_pred = X2_train_pred[X2] # Ordenar columnas según X2

m2 = MultiOutputRegressor(LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1))
# OJO: Entrenamos usando X2_train_pred (que contiene las predicciones de Y1), 
# pero el target sigue siendo la realidad (df_train[Y2])
m2.fit(X2_train_pred, df_train[Y2])

# Predecimos sobre el TRAIN y AJUSTAMOS antes de pasarlo al Nivel 3
Y2_train_pred_raw = pd.DataFrame(m2.predict(X2_train_pred), columns=Y2, index=df_train.index)
Y2_train_pred = ajustar_df_a_catalogos(Y2_train_pred_raw, MAPA_CATALOGOS)
Y2_train_pred['num_filas'] = Y1_train_pred['num_filas']
Y2_train_pred = aplicar_mascara_pasos(Y2_train_pred, col_num_filas='num_filas', evaluacion=False)
Y2_train_pred = Y2_train_pred.drop(columns=['num_filas'])

# --- NIVEL 3 ---
# Construimos X3 usando las PREDICCIONES AJUSTADAS de los Niveles 1 y 2
X3_train_pred = df_train[X1].copy()
X3_train_pred[Y1] = Y1_train_pred
X3_train_pred[Y2] = Y2_train_pred
X3_train_pred = X3_train_pred[X3] # Ordenar columnas según X3

m3 = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1)
m3.fit(X3_train_pred, df_train[Y3].values.ravel())

# Predecimos sobre el TRAIN y AJUSTAMOS antes de pasarlo al Nivel 4
Y3_train_pred_raw = pd.DataFrame(m3.predict(X3_train_pred), columns=Y3, index=df_train.index)
Y3_train_pred = ajustar_df_a_catalogos(Y3_train_pred_raw, MAPA_CATALOGOS)


# --- NIVEL 4 ---
# Construimos X4 usando las PREDICCIONES AJUSTADAS de los Niveles 1, 2 y 3
X4_train_pred = df_train[X1].copy()
X4_train_pred[Y1] = Y1_train_pred
X4_train_pred[Y2] = Y2_train_pred
X4_train_pred[Y3] = Y3_train_pred
X4_train_pred = X4_train_pred[X4] # Ordenar columnas según X4

m4 = MultiOutputRegressor(LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1))
m4.fit(X4_train_pred, df_train[Y4])

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,LGBMRegressor... verbosity=-1)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[LGBMRegressor... verbosity=-1), LGBMRegressor... verbosity=-1), LGBMRegressor... verbosity=-1), LGBMRegressor... verbosity=-1), ...]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[<U15](41,)","['Capacity_Moment','Capacity_Shear','Capacity_Axial',...,'fy_tornillo_mpa', 'fu_tornillo_mpa','fv_tornillo_mpa']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,41
,max_depth,6
,learning_rate,0.05
,n_estimators,200
,random_state,42
,n_jobs,-1


## Evaluación

In [ ]:
# =============================================================================
# 2. INFERENCIA EN CASCADA SOBRE VALIDACIÓN (Predict)
# =============================================================================

# --- NIVEL 1 ---
# Predicción de clasificación (num_filas, Bolt)
Y1_pred_val = pd.DataFrame(m1.predict(df_val[X1]), columns=Y1, index=df_val.index)


# --- NIVEL 2 ---
# 1. Construir features para Nivel 2 usando predicciones del Nivel 1
X2_val_pred = df_val[X1].copy()
X2_val_pred[Y1] = Y1_pred_val
X2_val_pred = X2_val_pred[X2] # Garantizar el orden exacto de las columnas

# 2. Predecir y Ajustar
Y2_pred_val_raw = pd.DataFrame(m2.predict(X2_val_pred), columns=Y2, index=df_val.index)
Y2_pred_val = ajustar_df_a_catalogos(Y2_pred_val_raw, MAPA_CATALOGOS)

# 3. Aplicar máscara de ceros según las filas predichas
Y2_pred_val['num_filas'] = Y1_pred_val['num_filas']
Y2_pred_val = aplicar_mascara_pasos(Y2_pred_val, col_num_filas='num_filas')
Y2_pred_val = Y2_pred_val.drop(columns=['num_filas'])


# --- NIVEL 3 ---
# 1. Construir features para Nivel 3 usando predicciones corregidas de Nivel 1 y 2
X3_val_pred = df_val[X1].copy()
X3_val_pred[Y1] = Y1_pred_val
X3_val_pred[Y2] = Y2_pred_val
X3_val_pred = X3_val_pred[X3]

# 2. Predecir y Ajustar
Y3_pred_val_raw = pd.DataFrame(m3.predict(X3_val_pred), columns=Y3, index=df_val.index)
Y3_pred_val = ajustar_df_a_catalogos(Y3_pred_val_raw, MAPA_CATALOGOS)


# --- NIVEL 4 ---
# 1. Construir features para Nivel 4 usando predicciones corregidas de Nivel 1, 2 y 3
X4_val_pred = df_val[X1].copy()
X4_val_pred[Y1] = Y1_pred_val
X4_val_pred[Y2] = Y2_pred_val
X4_val_pred[Y3] = Y3_pred_val
X4_val_pred = X4_val_pred[X4]

# 2. Predecir y Ajustar
Y4_pred_val_raw = pd.DataFrame(m4.predict(X4_val_pred), columns=Y4, index=df_val.index)
Y4_pred_val = ajustar_df_a_catalogos(Y4_pred_val_raw, MAPA_CATALOGOS)


# =============================================================================
# 3. EVALUACIÓN FINAL
# =============================================================================

# 1. Unir todas las predicciones limpias en un solo DataFrame
Y_val_pred_final = pd.concat([Y1_pred_val, Y2_pred_val, Y3_pred_val, Y4_pred_val], axis=1)

# 2. Seleccionar las mismas columnas del DataFrame real de validación
todas_las_Y = Y1 + Y2 + Y3 + Y4
Y_val_real = df_val[todas_las_Y]

# 3. Lanzar la evaluación (usando la función definida anteriormente)
resultados_df = evaluar_predicciones(Y_val_real, Y_val_pred_final)

                    EVALUACIÓN INTEGRAL DE PREDICCIONES Y TOLERANCIAS
 Variable  N (Muestras)  MAE (mm)  RMSE (mm)  R2 Score  % Exacto (Catálogo)  % <= 2mm (Taller)  % <= 5mm (Taller)
num_filas           945      0.08       0.38     0.945                 94.4               99.7              100.0
     Bolt           945      0.20       1.07     0.934                 95.9               96.3               98.8
        g           945      1.22       3.84     0.938                 84.2               84.2               95.9
        e           945      0.25       1.37     0.945                 96.2               96.2               98.9
        d           945      0.82       2.17     0.934                 72.3               89.7               96.6
       p1           945      1.97       4.93     0.957                 39.0               81.7               92.5
       p2           945      6.85      15.14     0.831                 17.2               41.7               68.9
       p3         